# Notebook 4 — Rescue, Reject & KNN Imputation

**Webinar 1: The AI-Ready Data Audit**

---

### Purpose

Clean, impute, and validate the dataset using three strategies:

| Strategy | When to Use | Applied To |
|----------|------------|------------|
| **Median** (Basic Rescue) | Supplementary numeric column, distribution permits central fill | `attendance_pct` |
| **Mode** (Basic Rescue) | Categorical column, most frequent value is a safe default | `city` |
| **KNN** (Advanced Rescue) | Column with segment-level significance, flat median destroys sub-group variance | `marks_math` |

### Input / Output

| | File |
|---|---|
| Input | `data/qa_checked_data.xlsx` |
| Output | `data/imputed_data.xlsx` |

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

DATA_DIR = os.path.join("..", "data")
input_file = os.path.join(DATA_DIR, "qa_checked_data.xlsx")

df = pd.read_excel(input_file, sheet_name="raw_data", engine="openpyxl")

print("=" * 60)
print("  NOTEBOOK 4 — RESCUE, REJECT & KNN IMPUTATION")
print("=" * 60)
print(f"\n✅ Loaded: {input_file}")
print(f"   Shape : {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# ===========================================================================
# Quality-Scoring Function (same 4 dimensions as Notebook 3)
# Defined here so we can compute BEFORE and AFTER scores.
# ===========================================================================

VALIDITY_RULES = {
    "marks_math":     (0, 100),
    "marks_science":  (0, 100),
    "attendance_pct": (0, 100),
}


def score_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """Return a scorecard DataFrame with one row per quality dimension."""

    # 1. Completeness
    completeness = round((1 - data.isnull().mean().mean()) * 100, 1)

    # 2. Validity
    validity_scores = []
    for col, (lo, hi) in VALIDITY_RULES.items():
        valid = data[col].notna()
        bad = data.loc[valid, col].apply(lambda v: v < lo or v > hi).sum()
        total = valid.sum()
        validity_scores.append(
            round((1 - bad / total) * 100, 1) if total else 100.0
        )
    validity = round(np.mean(validity_scores), 1)

    # 3. Uniqueness
    uniqueness = round((1 - data.duplicated().sum() / len(data)) * 100, 1)

    # 4. Consistency (city column)
    city_raw = data["city"].dropna()
    distinct_raw = city_raw.nunique()
    distinct_norm = city_raw.str.strip().str.title().nunique()
    consistency = round((distinct_norm / distinct_raw) * 100, 1) if distinct_raw else 100.0

    return pd.DataFrame([
        {"dimension": "Completeness", "score_pct": completeness},
        {"dimension": "Validity",     "score_pct": validity},
        {"dimension": "Uniqueness",   "score_pct": uniqueness},
        {"dimension": "Consistency",  "score_pct": consistency},
    ])


print("✅ score_dataset() defined — 4 quality dimensions.")

In [ ]:
# ===========================================================================
# BEFORE Scores (snapshot the raw data quality)
# ===========================================================================
before_scores = score_dataset(df)

print(f"{'=' * 60}")
print("BEFORE CLEANING — Quality Scores")
print("=" * 60)
print(before_scores.to_string(index=False))
print(f"Overall: {round(before_scores['score_pct'].mean(), 1)}%")

---

## Step 1: Fix Consistency & Drop Duplicates

We fix formatting issues BEFORE deduplication — normalising labels may
reveal duplicates that were hidden by string mismatches.

In [ ]:
# ===========================================================================
# Consistency Fix: canonical city mapping
# ===========================================================================
df_clean = df.copy()

city_map = {
    "delhi":      "Delhi",
    "mumbai":     "Mumbai",
    "bangalore":  "Bengaluru",
    "bengaluru":  "Bengaluru",
    "jaipur":     "Jaipur",
}

df_clean["city"] = (
    df_clean["city"]
    .str.strip()
    .str.lower()
    .map(city_map)
    .fillna(df_clean["city"].str.strip().str.title())
)

print("── Consistency Fix ──")
print(f"   Canonical cities: {sorted(df_clean['city'].unique())}")
print(f"   ✅ 'Bangalore' → 'Bengaluru', case/whitespace normalised.")

In [ ]:
# ===========================================================================
# Uniqueness Fix: drop exact duplicate rows
# ===========================================================================
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
rows_after = len(df_clean)

print(f"── Duplicate Removal ──")
print(f"   Rows before : {rows_before}")
print(f"   Rows after  : {rows_after}")
print(f"   Dropped     : {rows_before - rows_after}")
print(
    "\n   ✅ Data Leakage prevention: identical rows can no longer"
    "\n   appear in both Train and Test splits."
)

---

## Step 2: Outlier Capping (Validity Fix)

### 🏛️ Architectural Note: Hard-Capping vs. Model-Based Detection

The `clip()` we apply below is a **deterministic, interpretable rule** suitable
for pipelines where business stakeholders have defined explicit valid ranges
(e.g., "marks must be 0–100, attendance must be 0–100").

However, enterprise-grade outlier detection requires **context**. A student
with `attendance_pct = 98` and `marks_math = 12` isn't violating a range rule,
but the combination is anomalous (high attendance + very low marks). Detecting
this pattern requires model-based approaches:

- **K-Means Clustering** — segment students first, then define per-cluster norms
- **Isolation Forest** — learn the "normal" manifold, flag points that are easy to isolate
- **DBSCAN** — density-based approach that labels sparse-region points as anomalies

For today's webinar we use hard-capping as a pedagogical baseline.

In [ ]:
# ===========================================================================
# Validity Fix: cap impossible values to allowed range
# ===========================================================================

# marks_science: floor at 0
sci_bad = (df_clean["marks_science"] < 0).sum()
df_clean["marks_science"] = df_clean["marks_science"].clip(lower=0)

# attendance_pct: ceiling at 100
att_bad = (df_clean["attendance_pct"] > 100).sum()
df_clean["attendance_pct"] = df_clean["attendance_pct"].clip(upper=100)

print("── Outlier Capping ──")
print(f"   marks_science < 0  → capped to 0   ({sci_bad} row(s))")
print(f"   attendance_pct > 100 → capped to 100 ({att_bad} row(s))")
print(f"\n   Post-cap ranges:")
print(f"     marks_science  : [{df_clean['marks_science'].min():.0f}, {df_clean['marks_science'].max():.0f}]")
print(f"     attendance_pct : [{df_clean['attendance_pct'].min():.0f}, {df_clean['attendance_pct'].max():.0f}]")

---

## Step 3: Imputation — Rescue Strategies

### Missing Values Remaining

In [ ]:
print("── Missing Values Before Imputation ──")
missing = df_clean.isnull().sum()
print(missing[missing > 0].to_string())
if missing.sum() == 0:
    print("   No missing values remaining.")

### 3a. Median Rescue — `attendance_pct`

Attendance is a supplementary metric. Its distribution (post-capping)
is roughly uniform, so the median is a safe central-tendency fill.

In [ ]:
# ===========================================================================
# Method 1: MEDIAN imputation — attendance_pct
# ===========================================================================
att_median = df_clean["attendance_pct"].median()
att_missing = df_clean["attendance_pct"].isna().sum()
df_clean["attendance_pct"] = df_clean["attendance_pct"].fillna(att_median)

print(f"  [1] Median imputation → attendance_pct")
print(f"      Filled {att_missing} missing value(s) with median = {att_median}")

### 3b. Mode Rescue — `city`

City is categorical. The most frequent value is a reasonable default when
no other information is available.

In [ ]:
# ===========================================================================
# Method 2: MODE imputation — city
# ===========================================================================
city_missing = df_clean["city"].isna().sum()
if city_missing > 0:
    city_mode = df_clean["city"].mode()[0]
    df_clean["city"] = df_clean["city"].fillna(city_mode)
    print(f"  [2] Mode imputation → city")
    print(f"      Filled {city_missing} missing value(s) with mode = '{city_mode}'")
else:
    print(f"  [2] Mode imputation → city")
    print(f"      No missing values — skipped.")

### 3c. KNN Rescue — `marks_math`

### 🏛️ Architectural Deep-Dive: Why KNN, Not the Average

Plugging the global average math score into every missing cell **destroys the
distribution** of struggling vs. gifted students. If the average is 65, then
every student with a missing math score gets 65 — artificially inflating the
middle of the distribution and wiping out the tails.

For a downstream model predicting student drop-out risk, this is catastrophic:
the model loses its ability to distinguish at-risk students (low marks across
subjects) from high performers.

**KNN Imputation** solves this by finding the `k=5` students whose `marks_science`
and `attendance_pct` are most similar to the student with the missing math score.
It then infers the missing value from those neighbours' math scores.

The intuition: a student who scores 90 in science and has 95% attendance is
**behaviourally similar** to other high-performing students — their neighbours'
math scores are a far better proxy than the population average.

We are mathematically inferring the missing math score based strictly on
each student's other performance metrics.

In [ ]:
# ===========================================================================
# Method 3: KNN imputation — marks_math
# ===========================================================================
# Feature matrix: [marks_science, attendance_pct] → predict marks_math
# The imputer operates on the full matrix and fills NaN cells by averaging
# the values of the k nearest neighbours in feature space.

math_missing = df_clean["marks_math"].isna().sum()
print(f"\n  [3] KNN imputation → marks_math")
print(f"      Missing values : {math_missing}")
print(f"      Feature matrix : ['marks_science', 'attendance_pct', 'marks_math']")
print(f"      k (neighbours) : 5")

knn_cols = ["marks_science", "attendance_pct", "marks_math"]
imputer = KNNImputer(n_neighbors=5)
df_clean[knn_cols] = imputer.fit_transform(df_clean[knn_cols])

# Round marks to integers — fractional marks are nonsensical
df_clean["marks_math"] = df_clean["marks_math"].round(0).astype(int)

print(f"      ✅ KNN imputation complete.")
print(f"      Missing marks_math remaining: {df_clean['marks_math'].isna().sum()}")

In [ ]:
# ===========================================================================
# Final Missing-Value Check
# ===========================================================================
print("\n── Missing Values After Imputation ──")
remaining = df_clean.isnull().sum()
if remaining.sum() == 0:
    print("   ✅ All missing values filled — dataset is complete.")
else:
    print(remaining[remaining > 0].to_string())

---

## Before / After Comparison Scorecard

In [ ]:
# ===========================================================================
# AFTER Scores
# ===========================================================================
after_scores = score_dataset(df_clean)

print(f"{'=' * 60}")
print("AFTER CLEANING — Quality Scores")
print("=" * 60)
print(after_scores.to_string(index=False))
print(f"Overall: {round(after_scores['score_pct'].mean(), 1)}%")

In [ ]:
# ===========================================================================
# Side-by-Side Comparison
# ===========================================================================
comparison = before_scores.rename(columns={"score_pct": "before_pct"}).merge(
    after_scores.rename(columns={"score_pct": "after_pct"}),
    on="dimension",
)
comparison["change"] = (comparison["after_pct"] - comparison["before_pct"]).round(1)
comparison["change_str"] = comparison["change"].apply(
    lambda x: f"+{x}" if x > 0 else str(x)
)

before_overall = round(comparison["before_pct"].mean(), 1)
after_overall = round(comparison["after_pct"].mean(), 1)
overall_change = round(after_overall - before_overall, 1)

overall_row = pd.DataFrame([{
    "dimension":  "OVERALL",
    "before_pct": before_overall,
    "after_pct":  after_overall,
    "change":     overall_change,
    "change_str": f"+{overall_change}" if overall_change > 0 else str(overall_change),
}])
comparison = pd.concat([comparison, overall_row], ignore_index=True)

print(f"\n{'=' * 60}")
print("COMPARISON: Before vs After Cleaning")
print("=" * 60)
print(comparison[["dimension", "before_pct", "after_pct", "change_str"]].to_string(index=False))

print(
    f"\n🎯 Quality improvement: {before_overall}% → {after_overall}%"
    f" ({'+' if overall_change > 0 else ''}{overall_change} pts)"
)

In [ ]:
# ===========================================================================
# Export Final Clean Dataset
# ===========================================================================
output_file = os.path.join(DATA_DIR, "imputed_data.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="clean_data", index=False)
    comparison.to_excel(writer, sheet_name="score_comparison", index=False)
    before_scores.to_excel(writer, sheet_name="scores_before", index=False)
    after_scores.to_excel(writer, sheet_name="scores_after", index=False)

print(f"\n{'=' * 60}")
print("  NOTEBOOK 4 COMPLETE ✅  |  PIPELINE COMPLETE 🏁")
print("=" * 60)
print(f"\n   Exported: {output_file}")
print(f"   Sheets  : clean_data, score_comparison, scores_before, scores_after")
print(f"   Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(
    "\n   This dataset is now AI-ready:"
    "\n     • No missing values (Median, Mode, KNN imputed)"
    "\n     • No impossible values (hard-capped to valid ranges)"
    "\n     • No duplicates (Data Leakage eliminated)"
    "\n     • Consistent formatting (canonical city names)"
    "\n     • teacher_notes preserved for downstream RAG/LLM pipeline"
)